In [2]:
!nvidia-smi
!pip install -q transformers bitsandbytes accelerate pydantic


Wed Aug 12 13:01:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-1.7B"

print(f"Loading Base Model {MODEL_NAME} in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded successfully!")


Loading Base Model Qwen/Qwen3-1.7B in 4-bit...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Base model loaded successfully!


In [4]:
from google.colab import files
print("Upload your 'test_samples.json' file:")
uploaded = files.upload()


Upload your 'test_samples.json' file:


Saving test_samples.json to test_samples.json


In [5]:
SYSTEM_PROMPT = """You are a resume parser. Extract information from the resume text and return it as a single valid JSON object matching the required schema. Missing fields must be null (scalars) or [] (lists). Return ONLY the JSON. No markdown, no commentary."""

with open("test_samples.json", "r", encoding="utf-8") as f:
    test_samples = json.load(f)

before_results = []

print("Running BASE MODEL (Before Fine-Tuning) on 10 test samples...\n" + "="*60)

for sample in test_samples:
    s_id = sample["sample_id"]
    category = sample["category"]
    resume_text = sample["input"]

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nExtract candidate details from this resume:\n\n{resume_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    print(f"Sample {s_id} [{category}]: Finished")

    before_results.append({
        "sample_id": s_id,
        "category": category,
        "raw_resume_text": resume_text,
        "base_model_output_raw": raw_output,
        "ground_truth": sample["output"]
    })

# Save to JSON
output_file = "before_finetuning_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(before_results, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print(f"ALL 10 SAMPLES PROCESSED! Saved to '{output_file}'")


Running BASE MODEL (Before Fine-Tuning) on 10 test samples...
Sample 1 [Rich AI/ML Graduate]: Finished
Sample 2 [Senior DevOps (2 Phones)]: Finished
Sample 3 [Fresh Graduate (No Work Experience)]: Finished
Sample 4 [Career Changer (Mech to Data Science)]: Finished
Sample 5 [Missing Email Edge Case]: Finished
Sample 6 [Alternate Section Headers]: Finished
Sample 7 [PhD Researcher (Multiple Degrees)]: Finished
Sample 8 [Minimal Resume (Short Text)]: Finished
Sample 9 [Skills Buried in Projects (No Skills Section)]: Finished
Sample 10 [Inconsistent Date Formats]: Finished

ALL 10 SAMPLES PROCESSED! Saved to 'before_finetuning_results.json'


In [ ]:
from google.colab import files
files.download("before_finetuning_results.json")


In [6]:
# Install all required libraries
!pip install -q transformers datasets peft trl bitsandbytes accelerate pydantic pyyaml scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.6 MB/s eta 0:00:00


In [7]:
from google.colab import files
import os

print("1. Upload 'train.json' (70 examples):")
files.upload()

print("\n2. Upload 'val.json' (15 examples):")
files.upload()

print("\n3. Upload 'test_samples.json' (10 test samples):")
files.upload()



1. Upload 'train.json' (70 examples):


Saving train.json to train.json

2. Upload 'val.json' (15 examples):


Saving val.json to val.json

3. Upload 'test_samples.json' (10 test samples):


Saving test_samples.json to test_samples (1).json


{'test_samples (1).json': b'[\n  {\n    "sample_id": 1,\n    "category": "Rich AI/ML Graduate",\n    "description": "MSc AI & ML student with projects, internships, two degrees, and metric-rich bullets.",\n    "input": "ARJUN MEHRA\\nBengaluru, Karnataka | arjun.mehra.ai@gmail.com | +91 98765 41230\\nLinkedIn: linkedin.com/in/arjunmehra-ai | GitHub: github.com/arjunmehra\\n\\nOBJECTIVE\\nMSc Artificial Intelligence and Machine Learning student at IIIT Hyderabad with hands-on experience building RAG pipelines, anti-spoiler chatbots, and churn prediction systems. Proficient in Machine Learning, Deep Learning, LangChain, and Sentence Transformers.\\n\\nEDUCATION\\n- MSc Artificial Intelligence and Machine Learning | IIIT Hyderabad | 2024 \xe2\x80\x93 2026 | CGPA: 8.4\\n- BSc Computer Science | University of Delhi | 2021 \xe2\x80\x93 2024 | CGPA: 8.8\\n\\nEXPERIENCE\\nMachine Learning Intern | NeuralStack Technologies, Bengaluru | May 2025 \xe2\x80\x93 Jul 2025\\n- Developed intent classif

In [9]:
!pip install -q --upgrade --force-reinstall pyarrow datasets


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os._exit(0)


In [8]:
import os
os.environ["ACCELERATE_USE_BF16"] = "false"
os.environ["ACCELERATE_USE_FP16"] = "true"

import json
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

print(f"PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")

# 1. Load & Format Datasets
print("\nLoading datasets...")
raw_dataset = load_dataset("json", data_files={"train": "train.json", "val": "val.json"})

SYSTEM_PROMPT = """You are a resume parser. Extract information from the resume text and return it as a single valid JSON object matching the required schema.
Rules:
- Return ONLY the JSON object. No explanation, no markdown.
- If a field is missing from the resume, set it to null (scalars) or [] (lists).
- Never invent information that is not present in the resume."""

def build_prompt_text(example):
    inst = example.get("instruction", "Extract candidate details.")
    inp = example.get("input", "")
    out = json.dumps(example["output"], ensure_ascii=False) if isinstance(example["output"], dict) else str(example["output"])
    return {"text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{inst}\n\nResume Text:\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"}

train_dataset = raw_dataset["train"].map(build_prompt_text)
val_dataset   = raw_dataset["val"].map(build_prompt_text)

# 2. Load Base Model
MODEL_NAME = "Qwen/Qwen3-1.7B"
print(f"Loading {MODEL_NAME}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Prepare for QLoRA
base_model = prepare_model_for_kbit_training(base_model)

# 4. LoRA Config
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
)

# 5. SFTConfig
sft_config = SFTConfig(
    output_dir="./adapter_v1",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    save_total_limit=2,
    report_to="none",
    max_length=2048,
    dataset_text_field="text",
)

# 6. Create Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

# ============================================================
# KEY FIX: Cast ALL trainable adapter params to FLOAT32
# GradScaler (fp16 mode) requires gradients in float32.
# The forward pass runs in float16 (autocast), but optimizer
# states and gradients MUST be float32.
# ============================================================
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Verify
dtypes = set(p.dtype for p in trainer.model.parameters() if p.requires_grad)
print(f"\nAdapter param dtypes: {dtypes}")  # MUST show {torch.float32}

print("="*60)
print("Starting QLoRA Fine-Tuning (5 Epochs, ~20-25 mins)...")
print("="*60)

trainer.train()

# 7. Save
trainer.model.save_pretrained("./adapter_v1")
tokenizer.save_pretrained("./adapter_v1")
print("\nDone! Adapter saved to ./adapter_v1")


PyTorch: 2.11.0+cu128 | GPU: Tesla T4

Loading datasets...
Loading Qwen/Qwen3-1.7B...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Adapter param dtypes: {torch.float32}
Starting QLoRA Fine-Tuning (5 Epochs, ~20-25 mins)...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.356197,1.226950,1.415748,77962.000000,0.736480
2,0.864032,0.918429,0.863334,155924.000000,0.801733
3,0.763723,0.802741,0.790324,233886.000000,0.823615
4,0.635387,0.752359,0.709355,311848.000000,0.832625
5,0.671358,0.742195,0.696919,389810.000000,0.834590



Done! Adapter saved to ./adapter_v1


In [9]:
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Load Base Model + Fine-Tuned Adapter
MODEL_NAME = "Qwen/Qwen3-1.7B"
ADAPTER_PATH = "./adapter_v1"

print(f"Loading Base Model {MODEL_NAME} with Fine-Tuned Adapter...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Attach our trained LoRA Adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Run Test Samples
SYSTEM_PROMPT = """You are a resume parser. Extract information from the resume text and return it as a single valid JSON object matching the required schema.
Rules:
- Return ONLY the JSON object. No explanation, no markdown.
- If a field is missing from the resume, set it to null (scalars) or [] (lists).
- Never invent information that is not present in the resume."""

with open("test_samples.json", "r", encoding="utf-8") as f:
    test_samples = json.load(f)

after_results = []
print("\nRunning FINE-TUNED MODEL (AFTER Fine-Tuning) on 10 test samples...\n" + "="*60)

for sample in test_samples:
    s_id = sample["sample_id"]
    category = sample["category"]
    resume_text = sample["input"]

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nExtract candidate details:\n\n{resume_text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"Sample {s_id} [{category}] Done!")

    after_results.append({
        "sample_id": s_id,
        "category": category,
        "raw_resume_text": resume_text,
        "finetuned_model_output_raw": raw_output,
        "ground_truth": sample["output"]
    })

# Save to JSON
output_file = "after_finetuning_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(after_results, f, indent=2, ensure_ascii=False)

print("="*60)
print(f"SUCCESS! Saved AFTER fine-tuning outputs to '{output_file}'")


Loading Base Model Qwen/Qwen3-1.7B with Fine-Tuned Adapter...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


Running FINE-TUNED MODEL (AFTER Fine-Tuning) on 10 test samples...
Sample 1 [Rich AI/ML Graduate] Done!
Sample 2 [Senior DevOps (2 Phones)] Done!
Sample 3 [Fresh Graduate (No Work Experience)] Done!
Sample 4 [Career Changer (Mech to Data Science)] Done!
Sample 5 [Missing Email Edge Case] Done!
Sample 6 [Alternate Section Headers] Done!
Sample 7 [PhD Researcher (Multiple Degrees)] Done!
Sample 8 [Minimal Resume (Short Text)] Done!
Sample 9 [Skills Buried in Projects (No Skills Section)] Done!
Sample 10 [Inconsistent Date Formats] Done!
SUCCESS! Saved AFTER fine-tuning outputs to 'after_finetuning_results.json'
